# Finding Bottlenecks with PyTorch Profiler
---

In the previous notebook we measured our training loop and found it slower than expected.  In this notebook we will reach for our first profiling tool — `torch.profiler` — to find out *which section* of the training step is eating the time.

This notebook is **hands-on**: you will add profiling instrumentation yourself, observe what the profiler shows (and doesn't show), learn how to improve the profiler output, and finally fix the performance bug.

At each step there is a link to a reference solution if you get stuck.

## Baseline: How Slow Is It?

Let's re-run [`train_v1.py`](../source_code/intro/train_v1.py) to get a clean baseline measurement.  This is a straightforward CIFAR-10 + ResNet18 FP32 training loop on a single GPU — nothing exotic.

In [ ]:
!python ../source_code/intro/train_v1.py

**Expected output:**

```
steps timed: 55  mean step: ~260 ms  throughput: ~985 img/s
```

~985 images per second on a modern GPU.  That feels slow.  An NVIDIA L4 can push tens of thousands of images per second through ResNet18 at full utilisation.  Something is wrong — but the timing alone does not tell us what.

This is exactly when we reach for the profiler.

## Introducing `torch.profiler`

The PyTorch Profiler is a context manager that wraps your training loop and records the time spent in each operation.  The key API is:

```python
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=torch.profiler.schedule(wait=1, warmup=5, active=10, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler('/workspace/logs/my_run'),
    record_shapes=True,
) as prof:
    for step, (x, y) in enumerate(loader):
        # ... training step ...
        prof.step()   # ← tell the profiler a step just finished
```

**Key parameters:**

- `activities` — record both CPU-side Python calls and CUDA kernel launches
- `schedule` — skip the first `wait` steps, spend `warmup` steps calibrating, then actively record `active` steps; avoids capturing noisy early steps
- `on_trace_ready` — write a TensorBoard-compatible trace file when the active window closes
- `prof.step()` — **must** be called at the end of every step so the profiler knows where step boundaries are

The profiler adds some overhead (~5–10 %), so we only record a subset of steps rather than the whole run.

---
## Exercise 1: Add the Profiler Wrapper

Open [`train_v1.py`](../source_code/intro/train_v1.py) in the file browser and add the profiler wrapper around the training loop.

You will need to:

1. Add imports at the top (already available via `torch.profiler`)
2. Create a log directory for the trace output
3. Wrap the training loop in `torch.profiler.profile(...)`
4. Call `prof.step()` at the end of each iteration

Here is a skeleton to guide you:

```python
from pathlib import Path

logdir = Path("/workspace/logs/my_profiled_run")
logdir.mkdir(parents=True, exist_ok=True)

with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=torch.profiler.schedule(wait=1, warmup=5, active=10, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler(str(logdir)),
    record_shapes=True,
) as prof:
    for step in range(NUM_ITERS):
        # ... your existing training code ...
        prof.step()
```

**Stuck?** See the reference solution: [`train_v1_profile.py`](../source_code/intro/train_v1_profile.py)

Once you have added the profiler, run your modified script (or the reference solution):

In [ ]:
!python ../source_code/intro/train_v1_profile.py

**Expected output:**

```
steps timed: 55  mean step: ~278 ms  throughput: ~921 img/s
Trace written to /workspace/logs/train_v1_profile
```

The step time is slightly higher than the plain run — that is the profiler's own overhead.  The trace file has been written to `/workspace/logs/train_v1_profile/`.

## Viewing the Trace in TensorBoard

In [ ]:
from IPython.display import display, Javascript
display(Javascript("""
  var tb_url = 'https://' + window.location.hostname.replace('notebooks-', 'tensorboard-');
  element.innerHTML = '<b>TensorBoard:</b> <a href="' + tb_url + '" target="_blank">' + tb_url + '</a>';
"""))

Open TensorBoard using the link shown in the cell above.

Select the **PyTorch Profiler** plugin from the top navigation, then choose `train_v1_profile` from the run selector on the left.

Look at the **Overview** tab.  In the **Execution Summary** panel you will see a breakdown like this:

| Category | Time (μs) | % |
|----------|-----------|---|
| Kernel | ~56,000 | ~30% |
| CPU Exec | ~45,000 | ~24% |
| **Other** | **~84,000** | **~45%** |
| Memcpy | ~500 | <1% |
| DataLoader | 0 | 0% |

Wait — **DataLoader shows 0%**, but **Other is 45%**?  Where is all that time going?

## The Mystery of "Other"

The profiler tracks CUDA kernels and CPU operations, but it does not automatically know *which part of your code* each operation belongs to.  Without explicit labels, time spent loading data, running the forward pass, or computing gradients all gets lumped into generic categories or "Other".

To get a meaningful breakdown like "DataLoader: 90%, Forward: 5%, Backward: 4%", you need to **label each section** using `torch.profiler.record_function()`.

---
## Exercise 2: Add Section Labels with `record_function`

`torch.profiler.record_function(name)` is a context manager that labels a block of code.  The profiler will then report time spent in that block under the given name.

```python
with torch.profiler.record_function("DataLoader"):
    x, y = next(loader_iter)

with torch.profiler.record_function("H2D"):
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)

with torch.profiler.record_function("Forward"):
    logits = model(x)
    loss = loss_fn(logits, y)

with torch.profiler.record_function("Backward"):
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

with torch.profiler.record_function("Optimizer"):
    optimizer.step()
```

Add these labels to your profiled training script (or copy from [`train_v1_profile.py`](../source_code/intro/train_v1_profile.py) and add them).

**Stuck?** See the reference solution: [`train_v1_profile_labeled.py`](../source_code/intro/train_v1_profile_labeled.py)

Run the labeled version:

In [ ]:
!python ../source_code/intro/train_v1_profile_labeled.py

**Expected output:**

```
steps timed: 55  mean step: ~260 ms  throughput: ~985 img/s
Trace written to /workspace/logs/train_v1_profile_labeled
```

Now refresh TensorBoard and select `train_v1_profile_labeled`.

In the **Operator** view, you will now see your labeled sections.  Or look at the **Trace** view — zoom into a single step and you will see colored bars for DataLoader, Forward, Backward, and Optimizer.

**Now the picture is clear: DataLoader dominates each step.**  The GPU sits idle while the CPU loads and preprocesses each batch on a single thread before the next forward pass can begin.

## The Problem: Single-Threaded Data Loading

Look at these two settings in [`train_v1.py`](../source_code/intro/train_v1.py):

```python
NUM_WORKERS = 0      # ← Bug: data loading happens on the main thread
PIN_MEMORY  = False  # ← Bug: batches land in pageable memory
```

With `num_workers=0` every batch is decoded, augmented, and assembled by the same Python thread that runs the training loop.  The GPU has to wait for the CPU to finish before it can start the next forward pass.

With `pin_memory=False` the DataLoader allocates tensors in ordinary pageable memory.  Copying pageable memory to the GPU requires an intermediate staging buffer in pinned memory and cannot be overlapped with GPU compute.

**The fix is two lines:**

```python
NUM_WORKERS = 4      # spawn 4 worker processes to load data in parallel
PIN_MEMORY  = True   # allocate directly in pinned memory for fast H→D copies
```

Pytorch DataLoader workers run in separate processes, so they can decode and augment the *next* batch while the GPU is running the *current* forward pass — the CPU and GPU work in parallel instead of alternating.

---
## Exercise 3: Fix the DataLoader Bug

In your labeled script, change:

```python
NUM_WORKERS = 0       # → change to 4
PIN_MEMORY  = False   # → change to True
```

**Stuck?** See the reference solution: [`train_v1_fixed.py`](../source_code/intro/train_v1_fixed.py)

Run the fixed version:

In [ ]:
!python ../source_code/intro/train_v1_fixed.py

**Expected output:**

```
steps timed: 55  mean step: ~94 ms  throughput: ~2727 img/s
Trace written to /workspace/logs/train_v1_fixed
```

Refresh TensorBoard and select the `train_v1_fixed` run.  The step timeline should now look very different — DataLoader has shrunk to a small slice and GPU compute fills most of each step.

## Comparing Before and After

| Script | Mean step | Throughput | DataLoader share |
|---|---|---|---|
| [`train_v1.py`](../source_code/intro/train_v1.py) (buggy) | ~260 ms | ~985 img/s | ~90 % |
| [`train_v1_fixed.py`](../source_code/intro/train_v1_fixed.py) | ~94 ms | ~2727 img/s | small |

**~2.8× speedup from two lines of code.**

The profiler told us *which section* was slow.  The fix was straightforward once we knew where to look.

But this was a simple case — the profiler handed us the answer.  In the next notebook we will encounter a problem where the profiler shows us that something is wrong, but cannot tell us *what* or *why*.

---
## Recap: What We Learned

1. **Wall-clock timing** tells you something is slow, but not where.
2. **`torch.profiler`** records CPU and GPU activity and writes traces viewable in TensorBoard.
3. **Without labels**, time gets lumped into "Other" — the profiler can't read your mind.
4. **`record_function()`** labels sections so the profiler can show you a meaningful breakdown.
5. **DataLoader bottlenecks** are common — always use `num_workers > 0` and `pin_memory=True`.

> **Note:** We will follow this same hands-on pattern in the Nsight Systems notebook: you will add NVTX annotations yourself, see what they reveal, and fix the bug.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red; height:80px;"><b><br/>[Next Notebook — Profiler Exports for Batch Workflows](intro-profiler-exports.ipynb)</b></div></center>

---

## Links and Resources

- [PyTorch Profiler documentation](https://pytorch.org/docs/stable/profiler.html)
- [PyTorch Profiler with TensorBoard tutorial](https://pytorch.org/tutorials/intermediate/tensorboard_profiler_tutorial.html)
- [PyTorch Profiler recipe](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0).